In [0]:
# =============================================================
# audit_writer.py
# Layer   : Audit
# Purpose : Append-only Delta writers for standalone audit output.
#
# Output paths:
#   abfss://audit@petiot.dfs.core.windows.net/run_summary
#   abfss://audit@petiot.dfs.core.windows.net/table_validation_summary
#   abfss://audit@petiot.dfs.core.windows.net/rule_results
# =============================================================

from datetime import datetime, timezone
from pyspark.sql import functions as F


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def ensure_audit_container_access(storage_account: str = "petiot", audit_container: str = "audit") -> None:
    root = audit_path("", storage_account, audit_container)
    try:
        dbutils.fs.ls(root)
        print(f"[audit_writer] Audit container is accessible: {root}")
    except Exception as exc:
        raise RuntimeError(
            "Audit container is not accessible. Create the ADLS container first, "
            f"then grant the Databricks service principal access. Path: {root}. "
            f"Original error: {exc}"
        )


def _append_records(records: list[dict], relative_path: str, storage_account: str, audit_container: str) -> int:
    if not records:
        print(f"[audit_writer] No records to write for {relative_path}.")
        return 0

    target = audit_path(relative_path, storage_account, audit_container)
    df = spark.createDataFrame(records).withColumn("_audit_write_ts", F.current_timestamp())
    (
        df.write
          .format("delta")
          .mode("append")
          .option("mergeSchema", "true")
          .save(target)
    )
    print(f"[audit_writer] Wrote {len(records):,} records -> {target}")
    return len(records)


def write_run_summary(records: list[dict], storage_account: str = "petiot", audit_container: str = "audit") -> int:
    return _append_records(records, "run_summary", storage_account, audit_container)


def write_table_summary(records: list[dict], storage_account: str = "petiot", audit_container: str = "audit") -> int:
    return _append_records(records, "table_validation_summary", storage_account, audit_container)


def write_rule_results(records: list[dict], storage_account: str = "petiot", audit_container: str = "audit") -> int:
    return _append_records(records, "rule_results", storage_account, audit_container)


def read_audit_table(relative_path: str, storage_account: str = "petiot", audit_container: str = "audit"):
    return spark.read.format("delta").load(audit_path(relative_path, storage_account, audit_container))


print("[audit_writer] Loaded append-only audit writers.")